In [1]:

import os, json, time, math, glob, numpy as np, torch,torch.nn as nn,torch.nn.functional as F

In [2]:
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train_ids_bpe.npy", recursive=True)[0])
print(DATA_DIR, os.listdir(DATA_DIR))

train_ids = np.load(os.path.join(DATA_DIR, "train_ids_bpe.npy"), mmap_mode="r")
val_ids   = np.load(os.path.join(DATA_DIR, "val_ids_bpe.npy"),   mmap_mode="r")
VOCAB_SIZE = 65355                                  # 65,354 BPE tokens + 1 EOS
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"train {len(train_ids):,} | val {len(val_ids):,} | max id {int(val_ids.max())} | vocab {VOCAB_SIZE} | {device}")
print("first 10 train ids:", train_ids[:10].tolist())

/kaggle/input/datasets/ashwary007/phase1-arch2fyff ['val_ids_bpe.npy', 'train_ids_bpe.npy', 'legal_bpe_merges.json']
train 200,000,000 | val 5,000,000 | max id 65354 | vocab 65355 | cuda
first 10 train ids: [2967, 58, 46937, 25919, 61414, 17529, 20020, 48966, 6254, 22719]


In [3]:
class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

In [4]:
def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

In [5]:
def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

In [6]:
# 1. worked example: [1,0,1,0] at position 2 with d_head=4
cos4, sin4 = rope_tables(8, 4)
x = torch.zeros(1, 1, 3, 4); x[0, 0, 2] = torch.tensor([1., 0., 1., 0.])
print("worked example  :", apply_rope(x, cos4, sin4)[0, 0, 2])    # expect ≈ [-0.4161, 0.9093, 0.9998, 0.0200]

# 2. shapes
d_head = 16
cos, sin = rope_tables(256, d_head)
print("table shape     :", tuple(cos.shape))                       # expect (256, 8)

# 3. only the distance matters
q, k = torch.randn(d_head), torch.randn(d_head)
def at(v, pos):
    x = torch.zeros(1, 1, pos + 1, d_head); x[0, 0, pos] = v
    return apply_rope(x, cos, sin)[0, 0, pos]
a = (at(q, 5) @ at(k, 2)).item(); b = (at(q, 105) @ at(k, 102)).item(); c = (at(q, 5) @ at(k, 4)).item()
print(f"gap 3 near/far  : {a:.5f} vs {b:.5f} → equal: {abs(a - b) < 1e-4}")   # expect True
print(f"gap 1 (differs) : {c:.5f}")
print("length kept     :", torch.allclose(at(q, 50).norm(), q.norm()))      # expect True
print("pos 0 unchanged :", torch.allclose(at(q, 0), q))                     # expect True    # expect torch.Size([8, 2]) and ≈ [-0.4161, 0.9998]  (cos 2, cos 0.02)

worked example  : tensor([-0.4161,  0.9093,  0.9998,  0.0200])
table shape     : (256, 8)
gap 3 near/far  : -7.91713 vs -7.91713 → equal: True
gap 1 (differs) : -5.21901
length kept     : True
pos 0 unchanged : True


In [7]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

In [8]:
def forward(self,x):
    B,T,C=x.shape
    q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
    k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
    v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
    q=apply_rope(q,self.cos,self.sin)
    k=apply_rope(k,self.cos,self.sin)
    k=k.repeat_interleave(self.n_rep,dim=1)
    v=v.repeat_interleave(self.n_rep,dim=1)
    scores=(q*self.scale)@k.transpose(-2,-1)
    scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
    weights=self.attn_dropout(F.softmax(scores,dim=-1))
    out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
    return self.resid_dropout(self.out_proj(out))

In [9]:
torch.manual_seed(0)
B, T, d_model, H, Hkv = 2, 16, 64, 8, 2
att = GQAttention(d_model, H, Hkv, max_seq_len=32).eval()
x = torch.randn(B, T, d_model)
y = att(x)

print("output shape   :", tuple(y.shape))                         # expect (2, 16, 64)
print("k_proj outputs :", att.k_proj.out_features)                # expect 16  (2 kv heads × d_head 8)
n = sum(p.numel() for p in att.parameters())
print("params         :", n, "| expected 10240")                  # q 4096 + out 4096 + k 1024 + v 1024

x2 = x.clone(); x2[:, -1] += 1.0                                  # change only the LAST token
print("causal OK      :", torch.allclose(att(x2)[:, :-1], y[:, :-1], atol=1e-5))   # expect True

mha = GQAttention(d_model, H, H, max_seq_len=32)                  # n_kv_heads == n_heads → ordinary MHA
print("MHA-mode k_proj:", mha.k_proj.out_features)                # expect 64

output shape   : (2, 16, 64)
k_proj outputs : 16
params         : 10240 | expected 10240
causal OK      : True
MHA-mode k_proj: 64


In [10]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))
        

In [11]:
ff = SwiGLU(64, 172)
x = torch.randn(2, 5, 64)
y = ff(x)
print("output shape :", tuple(y.shape))                                       # expect (2, 5, 64)
print("params       :", sum(p.numel() for p in ff.parameters()), "| expected 33024")   # 3 × 64 × 172

with torch.no_grad():
    ff.w1.weight.zero_()                                                       # gate input → 0, so SiLU(0) = 0
print("gate closed → output zero:", torch.allclose(ff(x), torch.zeros_like(y)))   # expect True

output shape : (2, 5, 64)
params       : 33024 | expected 33024
gate closed → output zero: True


In [12]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class LlamaLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)
        

In [13]:
torch.manual_seed(0)
small = LlamaConfig(vocab_size=VOCAB_SIZE, d_model=64, n_layers=2, n_heads=8,
                    n_kv_heads=2, d_ff=172, max_seq_len=64)
m = LlamaLM(small).eval()
x = torch.randint(0, VOCAB_SIZE, (2, 32))
y = torch.randint(0, VOCAB_SIZE, (2, 32))
with torch.no_grad():
    logits = m(x)
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

print("logits shape :", tuple(logits.shape))                                   # expect (2, 32, 65355)
print(f"initial loss : {loss.item():.3f} | expected ≈ {math.log(VOCAB_SIZE):.3f}")
x2 = x.clone(); x2[:, -1] = (x2[:, -1] + 1) % VOCAB_SIZE
with torch.no_grad():
    print("causal OK    :", torch.allclose(m(x2)[:, :-1], logits[:, :-1], atol=1e-4))   # expect True
print("params       :", sum(p.numel() for p in m.parameters()), "| expected 4269568")
print("head tied    :", m.head.weight is m.tok_emb.weight)                      # expect True

logits shape : (2, 32, 65355)
initial loss : 11.114 | expected ≈ 11.088
causal OK    : True
params       : 4269568 | expected 4269568
head tied    : True


In [14]:
def count_llama(V, d, L, H, Hkv, d_ff):
    dh = d // H
    ok = (d % H == 0) and (dh % 2 == 0) and (H % Hkv == 0)
    attn = 2*d*d + 2*d*(Hkv*dh)           # q + out, plus the smaller k + v
    ffn  = 3*d*d_ff                        # SwiGLU: 3 matrices
    per_layer = attn + ffn + 2*d           # + 2 RMSNorms
    total = V*d + L*per_layer + d          # tied embedding/head + final norm
    print(f"d={d} L={L} H={H} Hkv={Hkv} d_ff={d_ff} | d_head={dh} | "
          f"embed {V*d/1e6:.1f}M | layers {L*per_layer/1e6:.1f}M | total {total/1e6:.1f}M | "
          f"{'OK' if ok and 50e6 <= total <= 300e6 else 'CHECK'}")

count_llama(65355,1024, 10, 16, 8,2816)       # sanity: must say total 4.3M (matches the test)
# then try your own candidates, e.g.
# count_llama(65355, d, L, H, Hkv, d_ff)

d=1024 L=10 H=16 Hkv=8 d_ff=2816 | d_head=64 | embed 66.9M | layers 118.0M | total 184.9M | OK


In [15]:
cfg = LlamaConfig(vocab_size=VOCAB_SIZE, d_model=1024, n_layers=10, n_heads=16,
                  n_kv_heads=8, d_ff=2816, max_seq_len=1024, dropout=0.1)   # your choice (arch 1 used 0.1)
model = LlamaLM(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")         # expect 184,909,824

model.eval()
x = torch.randint(0, VOCAB_SIZE, (2, 256), device=device)
y = torch.randint(0, VOCAB_SIZE, (2, 256), device=device)
with torch.no_grad():
    logits = model(x)
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
print(f"initial loss: {loss.item():.3f} | expected ≈ {math.log(VOCAB_SIZE):.3f}")
x2 = x.clone(); x2[:, -1] = (x2[:, -1] + 1) % VOCAB_SIZE
with torch.no_grad():
    print("causal mask OK:", torch.allclose(logits[:, :-1], model(x2)[:, :-1], atol=1e-4))
model.train()

184,909,824 parameters
initial loss: 11.231 | expected ≈ 11.088
causal mask OK: True


LlamaLM(
  (tok_emb): Embedding(65355, 1024)
  (blocks): ModuleList(
    (0-9): 10 x LlamaBlock(
      (norm1): RMSNorm()
      (attn): GQAttention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (k_proj): Linear(in_features=1024, out_features=512, bias=False)
        (v_proj): Linear(in_features=1024, out_features=512, bias=False)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): RMSNorm()
      (ffn): SwiGLU(
        (w1): Linear(in_features=1024, out_features=2816, bias=False)
        (w3): Linear(in_features=1024, out_features=2816, bias=False)
        (w2): Linear(in_features=2816, out_features=1024, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm_f): RMSNorm()
  (head): Linear(in_features=1024, out_features=65355, bias=False)
)

In [16]:
from dataclasses import asdict

MICRO_B = 4
ACCUM   = 122
MAX_STEPS = 400          # placeholder: set the real value after the smoke test

PEAK_LR = 6.0e-4
MIN_LR  = 6.0e-5
WARMUP  = 40

WEIGHT_DECAY = 0.1
CLIP         = 1.0

EVAL_EVERY   = 25
EVAL_BATCHES = 10
LOG_EVERY    = 5

CKPT_PATH  = "/kaggle/working/arch2_ckpt.pt"
FINAL_PATH = "/kaggle/working/arch2_final.pt"
TIME_LIMIT = 11 * 3600

T = cfg.max_seq_len

assert WARMUP < MAX_STEPS, "warmup must be shorter than training"
print(f"tokens per optimizer step: {MICRO_B * ACCUM * T:,}")

tokens per optimizer step: 499,712


In [17]:
decay_params    = [p for _, p in model.named_parameters() if p.dim() >= 2]
no_decay_params = [p for _, p in model.named_parameters() if p.dim() < 2]

optimizer = torch.optim.AdamW([
    {"params": decay_params,    "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0}
], lr=PEAK_LR)

def get_lr(step):
    if step < WARMUP:
        return PEAK_LR * (step + 1) / WARMUP
    if step >= MAX_STEPS:
        return MIN_LR
    decay_ratio = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * decay_ratio))

scaler = torch.amp.GradScaler("cuda")

num_decay    = sum(p.numel() for p in decay_params)
num_no_decay = sum(p.numel() for p in no_decay_params)
print(f"decay {num_decay:,} | no-decay {num_no_decay:,} | total {num_decay + num_no_decay:,} (want 184,909,824)")
print(f"lr: start {get_lr(0):.2e} | peak {get_lr(WARMUP):.2e} | end {get_lr(MAX_STEPS):.2e}")

decay 184,888,320 | no-decay 21,504 | total 184,909,824 (want 184,909,824)
lr: start 1.50e-05 | peak 6.00e-04 | end 6.00e-05


In [18]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            x, y = get_batch(split, MICRO_B, T)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

print(estimate_loss())

NameError: name 'get_batch' is not defined

In [ ]:
history  = {"step": [], "train": [], "val": [], "lr": []}
best_val = float("inf")
start_step = 0
tokens_per_step = MICRO_B * ACCUM * T
t0 = t_log = time.time()
model.train()

def save(step, tag, path=CKPT_PATH):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "step": step, "cfg": asdict(cfg),
                "best_val": best_val, "history": history}, path)
    print(f"  saved checkpoint ({tag}) at step {step} -> {path}")

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    for _ in range(ACCUM):
        x, y = get_batch("train", MICRO_B, T)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1)) / ACCUM
        scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
    scaler.step(optimizer); scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if not math.isfinite(loss.item()):
        print(f"  NaN/inf loss at step {step}: stopping. Last good checkpoint is safe.")
        break
    if step % LOG_EVERY == 0:
        torch.cuda.synchronize()
        dt = time.time() - t_log; t_log = time.time()
        n = LOG_EVERY if step > 0 else 1
        print(f"step {step:5d} | loss {loss.item()*ACCUM:.3f} | lr {lr:.2e} | gnorm {gnorm:.2f} | "
              f"{tokens_per_step*n/dt:,.0f} tok/s | peak mem {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
    if (step > 0 and step % EVAL_EVERY == 0) or step == MAX_STEPS - 1:
        l = estimate_loss()
        for k, v in (("step", step), ("train", l["train"]), ("val", l["val"]), ("lr", lr)):
            history[k].append(v)
        print(f"  EVAL step {step} | train {l['train']:.3f} | val {l['val']:.3f} | val ppl {math.exp(l['val']):.1f}")
        if l["val"] < best_val:
            best_val = l["val"]; save(step, "best")
        t_log = time.time()
    if time.time() - t0 > TIME_LIMIT:
        print(f"  time limit reached at step {step}"); break

print(f"done in {(time.time()-t0)/60:.1f} min | best val {best_val:.3f}")
save(step, "final", FINAL_PATH)
with open("/kaggle/working/arch2_history.json", "w") as f:
    json.dump(history, f)